# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *



In [2]:
#load data
model_name = "llama3-70b-8192"
fnout = f"llm_response_impact_2rep_test_{model_name.replace('/', '_')}.csv"
response_df = pd.read_csv(DATA_OUT_LLMS+fnout)

In [3]:
response_df

,impactType,impactValue,impactUnit,impactValueFlag,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,impactAnnotation,appealCode,country,reportDate,disasterType
0,Affected People,2600000.0,people,exact,['southern Bangladesh'],NaN,NaN,NaN,NaN,NaN,NaN,"['cyclonic storm', 'flooding']",['The cumulative effect of the floods coming a...,MDRBD015,Bangladesh,2016-03-04 00:00:00,Cyclone
1,Affected People,218665.0,people,exact,['Bangladesh'],NaN,NaN,NaN,NaN,NaN,NaN,"['cyclonic storm', 'flooding']","['It is estimated that 218,665 people (57,774 ...",MDRBD015,Bangladesh,2016-03-04 00:00:00,Cyclone
2,Agriculture,NaN,NaN,NaN,['Bangladesh'],NaN,NaN,NaN,NaN,NaN,NaN,"['cyclonic storm', 'flooding']",['Crops were damaged and shrimp projects flood...,MDRBD015,Bangladesh,2016-03-04 00:00:00,Cyclone
3,Residential Buildings,NaN,NaN,NaN,['Bangladesh'],NaN,NaN,NaN,NaN,NaN,NaN,"['cyclonic storm', 'flooding']","['Many houses were flattened or under water, t...",MDRBD015,Bangladesh,2016-03-04 00:00:00,Cyclone
4,WASH infrastructure,NaN,NaN,NaN,['Bangladesh'],NaN,NaN,NaN,NaN,NaN,NaN,"['cyclonic storm', 'flooding']",['Power supplies and communication systems dis...,MDRBD015,Bangladesh,2016-03-04 00:00:00,Cyclone
5,Affected People,7600000.0,people,exact,['Bangladesh'],2019.0,7.0,NaN,NaN,NaN,NaN,"['landslide', 'flood']",['The heavy rainfall occurred during July to S...,MDRBD022,Bangladesh,2020-12-05 00:00:00,['Flood']
6,Displaced People,300000.0,people,exact,['Bangladesh'],2019.0,7.0,NaN,NaN,NaN,NaN,"['landslide', 'flood']",['According to the National Needs Assessment W...,MDRBD022,Bangladesh,2020-12-05 00:00:00,['Flood']
7,Human Deaths,114.0,people,exact,['Bangladesh'],2019.0,7.0,NaN,NaN,NaN,NaN,"['landslide', 'flood']",NaN,MDRBD022,Bangladesh,2020-12-05 00:00:00,['Flood']
8,Agriculture,532000.0,hectares,exact,['Bangladesh'],2019.0,7.0,NaN,NaN,NaN,NaN,"['landslide', 'flood']","['On top of that, according to the media, abou...",MDRBD022,Bangladesh,2020-12-05 00:00:00,['Flood']
9,Affected People,40000.0,people,exact,"['Rajshahi', 'Shariatpur', 'Kushtia', 'Rajbari...",2019.0,10.0,NaN,NaN,NaN,NaN,['flood'],"['Due to this second spell of flood, around 40...",MDRBD022,Bangladesh,2020-12-05 00:00:00,['Flood']


In [4]:
#convert numerical columns
num_cols = ["impactValue", "startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols)


In [5]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)